## Imports

In [ ]:
import zipfile 

import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 

from sklearn.base import BaseEstimator, TransformerMixin 
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler 
from sklearn.linear_model import LogisticRegression 
from sklearn.ensemble import HistGradientBoostingClassifier 

from utility.utils import GroupedDataSplitter, WindowTransformer, target_windower

## Data Extration from the raw files

In [ ]:
with zipfile.ZipFile('../data/archive.zip', 'r') as f:
    print(f'The archive contains: \n{f.namelist()}')
    f.extractall(path='../data/')

In [ ]:
dataset = pd.read_csv('../data/adhdata.csv')

## Variable declaration

In [ ]:
group_column = "ID" # used for subject-aware windowing, that is, based on individual subjects
target_column = "Class" 
window_size = 512 # number of rows in each window
train_ratio = 0.8 # fraction of the whole dataset in the training set

## Train-Test split by subjects

In [ ]:
splitter = GroupedDataSplitter(
    group_column=group_column,
    train_ratio=train_ratio,
    shuffle=True,
    random_state=42,
)

train_data, test_data = splitter.split(dataset)

In [ ]:
# window-level targets for the train and test set

y_train = target_windower(
    train_data,
    group_column=group_column,
    target_column=target_column,
    window_size=window_size,
)

y_test = target_windower(
    test_data,
    group_column=group_column,
    target_column=target_column,
    window_size=window_size,
)

In [ ]:
# Feature Matrix for train and test set containing the ID column

X_train = train_data.drop(columns=[target_column])
X_test = test_data.drop(columns=[target_column])

## Feature Engineering

## Modeling

### Model fitting

In [ ]:
# Model pipeline

linear_model = Pipeline([
    ("window", WindowTransformer(window_size=512, group_column="ID")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])

linear_model

In [ ]:
linear_model.fit(X_train, y_train)
linear_preds = linear_model.predict(X_test)

In [ ]:
ensemble_model = Pipeline([
    ("window", WindowTransformer(window_size=512, group_column="ID")),
    ("scaler", StandardScaler()),
    ("clf", HistGradientBoostingClassifier()),
])

ensemble_model

In [ ]:
ensemble_model.fit(X_train, y_train)
ensemble_preds = ensemble_model.predict(X_test)

### Model Evaluation